# Prepare shared DREAM3 in-silico gene-network data

This notebook creates or verifies the frozen `dream3` bundles used by
all eight method notebooks in this directory. Data generation and acquisition
are kept separate from model fitting so every method reads byte-identical
observations and the same declared ground truth.

DREAM3 Size 100 is an in-silico GeneNetWeaver benchmark, not a real-world measurement dataset. The CUTS+ repository references DREAM3 files that it does not distribute. This notebook therefore validates a user-supplied, licensed copy instead of downloading an unverified mirror.

In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import math
import os
import platform
import subprocess
import sys
import tempfile
import types
import urllib.request
import zipfile
import pickle
from pathlib import Path

EXPECTED_PYTHON = (3, 14)
if sys.version_info[:2] != EXPECTED_PYTHON:
    raise RuntimeError(
        "This notebook requires CPython 3.14; "
        f"found {platform.python_version()}."
    )

import numpy as np
import pandas as pd
import scipy.special

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "src" / "fastCGC" / "__init__.py").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the fastC-GC project root.")
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from fastCGC.experiments.upstream import verify_upstream_revision

In [ ]:
def series_digest(series):
    return hashlib.sha256(
        np.ascontiguousarray(series, dtype=np.float64).tobytes()
    ).hexdigest()


def save_bundle(
    path,
    *,
    series,
    metadata,
    aggregate_truth=None,
    lag_truth=None,
    reference_graph=None,
    extra_arrays=None,
):
    graph_arrays = (aggregate_truth, lag_truth, reference_graph)
    if sum(value is not None for value in graph_arrays) > 1:
        raise ValueError("Provide at most one truth or reference graph.")
    path.parent.mkdir(parents=True, exist_ok=True)
    arrays = {"series": np.asarray(series, dtype=float)}
    if lag_truth is not None:
        arrays["lag_truth"] = np.asarray(lag_truth, dtype=bool)
    elif aggregate_truth is not None:
        arrays["aggregate_truth"] = np.asarray(
            aggregate_truth,
            dtype=bool,
        )
    elif reference_graph is not None:
        arrays["reference_graph"] = np.asarray(
            reference_graph,
            dtype=bool,
        )
    for name, value in (extra_arrays or {}).items():
        if name in arrays:
            raise ValueError(f"Duplicate bundle array name: {name}.")
        arrays[name] = np.asarray(value)
    digest = series_digest(arrays["series"])
    metadata = {**metadata, "data_digest": digest}
    metadata_path = path.with_suffix(".json")
    if path.exists() or metadata_path.exists():
        if not path.is_file() or not metadata_path.is_file():
            raise FileNotFoundError(f"Incomplete dataset bundle at {path}.")
        recorded = json.loads(metadata_path.read_text(encoding="utf-8"))
        with np.load(path, allow_pickle=False) as stored:
            saved = {name: np.asarray(stored[name]) for name in stored.files}
        arrays_match = set(saved) == set(arrays) and all(
            np.array_equal(saved[name], value)
            for name, value in arrays.items()
        )
        if not arrays_match or recorded != metadata:
            raise ValueError(f"Existing bundle does not match request: {path}.")
        return False

    with tempfile.NamedTemporaryFile(
        mode="wb", dir=path.parent, prefix=f".{path.name}.", delete=False
    ) as handle:
        np.savez_compressed(handle, **arrays)
        temporary_data = Path(handle.name)
    os.replace(temporary_data, path)

    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        dir=path.parent,
        prefix=f".{metadata_path.name}.",
        delete=False,
    ) as handle:
        json.dump(metadata, handle, indent=2, sort_keys=True)
        handle.write("\n")
        temporary_metadata = Path(handle.name)
    os.replace(temporary_metadata, metadata_path)
    return True


def file_digest(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_verified(url, path, expected_sha256):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.is_file():
        observed = file_digest(path)
        if observed != expected_sha256:
            raise ValueError(
                f"File digest mismatch for {path}: {observed}."
            )
        return False

    with tempfile.NamedTemporaryFile(
        mode="wb",
        dir=path.parent,
        prefix=f".{path.name}.",
        delete=False,
    ) as handle:
        temporary = Path(handle.name)
        with urllib.request.urlopen(url, timeout=120) as response:
            while chunk := response.read(1024 * 1024):
                handle.write(chunk)
    try:
        observed = file_digest(temporary)
        if observed != expected_sha256:
            raise ValueError(
                f"Downloaded file digest mismatch: {observed}."
            )
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()
    return True

In [ ]:
DATASET_KEY = "dream3"
DATASET_DIR = (
    PROJECT_ROOT / "data" / "shared" / "causal-comparison"
    / DATASET_KEY / "datasets"
)
UNN_ROOT = Path(
    os.environ.get("UNN_ROOT", PROJECT_ROOT / "external" / "UNN")
).expanduser()
UPSTREAM_REVISION = verify_upstream_revision(
    UNN_ROOT,
    expected_url="https://github.com/jarrycyx/UNN.git",
    expected_commit="2212724cce8a7752db564089a7537fecedb0e36d",
)
CUTS_ROOT = UNN_ROOT / "CUTS_Plus"
if str(CUTS_ROOT) not in sys.path:
    sys.path.insert(0, str(CUTS_ROOT))
from data.simu_data import load_dream_data

# CUTS+ delegates this loader to the SRU_for_GCI layout, but the pinned UNN
# checkout does not contain those files. Fetch the exact preprocessed tensors
# and reference networks used by that loader from the original authors' pinned
# repository revision, verifying every file before use.
DREAM_DATA_ROOT = (
    UNN_ROOT / "causal_discov_sota" / "SRU_for_GCI" / "data" / "dream3"
)
DREAM3_FILES = {'Dream3TensorData/Size100Ecoli1.pt': 'c312c1a4d2570bfb0fbeae075868de9547f50efd70c2903697009586c5b70c97', 'Dream3TensorData/Size100Ecoli2.pt': '77d7cf39a0e78396f73922f3345b1650d99cd1445173c4bb87a2c86ee41e04a5', 'Dream3TensorData/Size100Yeast1.pt': '94844418b42c262ca9f378e673f124ec982cd0b635260c200f56f886091b9d3b', 'Dream3TensorData/Size100Yeast2.pt': '29e81a6e98b6b190590eed911d7e13c8e68ada53eba668dca6cb5bec9973c9ec', 'Dream3TensorData/Size100Yeast3.pt': 'ba509214bb86fa1d4a8160f95b0d384182aab38afed7545d47aaee09bc73d36b', 'TrueGeneNetworks/InSilicoSize100-Ecoli1.tsv': 'a3ff157a8398f52a13dd26af8407817ec81c0ac8d4582ccb38534e384f68cd73', 'TrueGeneNetworks/InSilicoSize100-Ecoli2.tsv': 'f5448add4389180a105e2e75f730278cf2d946b6948823091569f386fe97fde1', 'TrueGeneNetworks/InSilicoSize100-Yeast1.tsv': '00cebe38ad7a596c605dcc8db021f1b23e3ac309bc4f42775ef2490515212712', 'TrueGeneNetworks/InSilicoSize100-Yeast2.tsv': '7320839722d86d161522f9343d80513a8ee94dda38ffad823b4e668ae467ccae', 'TrueGeneNetworks/InSilicoSize100-Yeast3.tsv': '296e70fd0ee2aacb20b1079a8a070b0f633a421384b32f009011b75b0b75746d'}
DREAM3_RAW_ROOT = (
    "https://raw.githubusercontent.com/sakhanna/SRU_for_GCI/7fe4b24880a947056d7753cf51f69d5e4f0ee5fb/data/dream3"
)
for relative_path, expected_sha256 in DREAM3_FILES.items():
    download_verified(
        f"{DREAM3_RAW_ROOT}/{relative_path}",
        DREAM_DATA_ROOT / relative_path,
        expected_sha256,
    )

DATASET_IDS = tuple(range(5))
FIT_ORDER = 1
MAX_NEW_DATASETS: int | None = None

created = 0
original_cwd = Path.cwd()
try:
    os.chdir(UNN_ROOT)
    for dataset_id_number in DATASET_IDS:
        if MAX_NEW_DATASETS is not None and created >= MAX_NEW_DATASETS:
            break
        data, target_source_truth = load_dream_data(dataset_id_number)
        dataset_id = f"dream3-size100-{dataset_id_number}"
        was_created = save_bundle(
            DATASET_DIR / f"{dataset_id}.npz",
            series=np.asarray(data, dtype=float).T,
            aggregate_truth=np.asarray(target_source_truth, dtype=bool).T,
            metadata={
                "dataset_id": dataset_id,
                "dataset": DATASET_KEY,
                "loader": "UNN/CUTS_Plus/data/simu_data.py:load_dream_data",
                "upstream_commit": UPSTREAM_REVISION.commit,
                "data_source": (
                    "https://github.com/sakhanna/SRU_for_GCI/tree/"
                    "7fe4b24880a947056d7753cf51f69d5e4f0ee5fb/data/dream3"
                ),
                "data_source_commit": "7fe4b24880a947056d7753cf51f69d5e4f0ee5fb",
                "dataset_index": dataset_id_number,
                "fit_order": FIT_ORDER,
                "regime": "dream3-size100",
                "series_layout": "variable, time",
                "truth_layout": "source, target",
                "truth_scope": "aggregate directed gene-regulatory graph",
            },
        )
        created += int(was_created)
finally:
    os.chdir(original_cwd)
print(f"Created {created} new DREAM3 bundles. Rerun to resume.")